# Janis + Agilent E4890A Control Notebook

Quick testing notebook for impedance measurements with temperature and bias control.

**Setup required:**
1. Run `pip install -e .` in `instrument-interfaces/` directory (one time only)
2. Check GPIB addresses match your hardware
3. Run cells sequentially

In [ ]:
# Imports
from nfoinstruments import MeasurementSetup, E4890A, Janis
from nfoinstruments.procedures import sweep_frequency_agilent_format
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from time import sleep, time

# Counter for run numbering
run_count = 1

In [ ]:
# Imports
from nfoinstruments.drivers import Janis, E4890A
from nfoinstruments.drivers.setup import MeasurementSetup
from nfoinstruments.procedures import sweep_frequency_lcr
import numpy as np
import time

# Initialize instruments
mm = MeasurementSetup()

# Connect to devices (update GPIB addresses if needed)
mm.connect_to_devices({
    'GPIB0::16::INSTR': Janis,
    'GPIB0::17::INSTR': E4890A
})

janis = mm.devices['GPIB0::16::INSTR']
lcr = mm.devices['GPIB0::17::INSTR']

print(f"✓ Connected - Janis: {janis.temperature:.1f} K, LCR: Ready")

['GPIB0::15::INSTR', 'GPIB0::18::INSTR', 'GPIB0::22::INSTR', 'GPIB0::16::INSTR', 'GPIB0::17::INSTR']
{'_alc_enabled': True,
 '_averages': 1,
 '_bias': 0,
 '_frequency': 100,
 '_measurement_time': <MeasurementTime.MEDIUM: 'MED'>,
 '_measurement_type': <MeasurementType.RX: 'RX'>,
 '_signal_amplitude': 1,
 '_signal_type': <SignalType.VOLTAGE: 2>,
 'address': 'GPIB0::17::INSTR',
 'measurement_timeout': 3,
 'resource': <'GPIBInstrument'('GPIB0::17::0::INSTR')>}


In [ ]:
# Configure LCR meter
lcr.measurement_time = E4890A.MeasurementTime.MEDIUM
lcr.measurement_type = E4890A.MeasurementType.ZTD
lcr.signal_type = E4890A.SignalType.VOLTAGE
lcr.signal_amplitude = 0.05  # 50 mV
lcr.averages = 5
lcr.bias = 0
lcr.alc_enabled = True
lcr.cable_length = 0

print(f"✓ LCR configured: {lcr.measurement_type.name}, {lcr.signal_amplitude}V, {lcr.averages} avg")

In [ ]:
# Define frequency sweep points
frequency_points = np.round(
    np.logspace(np.log10(20), np.log10(2_000_000), num=100), 
    decimals=2
)

print(f"✓ Frequency sweep: {len(frequency_points)} points from {frequency_points[0]} Hz to {frequency_points[-1]} Hz")

---
## Option 1: Single Temperature Measurement

In [ ]:
# Single temperature measurement
target_temp = 300  # K

print(f"Setting temperature to {target_temp} K...")
janis.temperature_setpoint = target_temp

print("Waiting for stability...")
while not janis.temperature_stable:
    print(f"  Current: {janis.temperature:.1f} K")
    time.sleep(10)

print(f"✓ Stable at {janis.temperature:.1f} K")

# Filename format: run###_temp_###.csv (matches your import regex)
filename = f"C:/Users/F110216/Documents/DATA/Horatio/run{run_count}_temp_{janis.temperature:.0f}.csv"
print(f"Measuring to: {filename}")

with open(filename, "w") as f:
    # Write header (commented for reference, actual data format below)
    f.write("# time,bias,frequency,NA,Z,theta\n")
    sweep_frequency_lcr(janis, lcr, frequency_points, f)

run_count += 1
print(f"✓ Complete! Next run: {run_count}")

AttributeError: 'MeasurementSetup' object has no attribute 'janis'

---
## Option 2: Temperature Sweep

In [ ]:
# Temperature sweep (multiple temperatures)
temp_points = [300, 280, 260, 240, 220, 200]  # K

for target_temp in temp_points:
    print(f"\n{'='*50}")
    print(f"Temperature: {target_temp} K")
    print('='*50)
    
    janis.temperature_setpoint = target_temp
    print("Waiting for stability...")
    
    while not janis.temperature_stable:
        print(f"  Current: {janis.temperature:.1f} K")
        time.sleep(10)
    
    actual_temp = janis.temperature
    print(f"✓ Stable at {actual_temp:.1f} K")
    
    # Filename format: run###_temp_###.csv
    filename = f"C:/Users/F110216/Documents/DATA/Horatio/run{run_count}_temp_{actual_temp:.0f}.csv"
    print(f"Measuring to: {filename}")
    
    with open(filename, "w") as f:
        f.write("# time,bias,frequency,NA,Z,theta\n")
        sweep_frequency_lcr(janis, lcr, frequency_points, f)

---
## Option 3: Temperature + Bias Sweep

In [ ]:
# Temperature + Bias sweep
temp_points = [300, 250, 200]  # K
bias_points = np.linspace(0, 1, 6)  # 0 to 1V in 0.2V steps

for target_temp in temp_points:
    print(f"\n{'='*50}")
    print(f"Temperature: {target_temp} K")
    print('='*50)
    
    janis.temperature_setpoint = target_temp
    print("Waiting for stability...")
    
    while not janis.temperature_stable:
        print(f"  Current: {janis.temperature:.1f} K")
        sleep(10)
    
    actual_temp = janis.temperature
    print(f"✓ Stable at {actual_temp:.1f} K")
    
    for bias in bias_points:
        print(f"\n  Bias: {bias:.2f} V")
        lcr.bias = bias
        sleep(0.5)
        
        # Filename format: run###_temp_###_bias_#.##.csv (matches your import regex)
        filename = f"C:/Users/F110216/Documents/DATA/Horatio/run{run_count}_temp_{actual_temp:.0f}_bias_{bias:.2f}.csv"
        
        with open(filename, "w") as f:
            f.write("# time,bias,frequency,NA,Z,theta\n")
            sweep_frequency_agilent_format(janis, lcr, frequency_points, f)
        
        print(f"  ✓ Saved: run{run_count}")
        run_count += 1

print(f"\n{'='*50}")
print(f"ALL DONE! Next run: {run_count}")
print('='*50)

In [ ]:
# Load a single measurement file
import glob

# Find latest file
data_dir = "C:/Users/F110216/Documents/DATA/Horatio"
files = sorted(glob.glob(f"{data_dir}/run*.csv"))

if files:
    latest_file = files[-1]
    print(f"Loading: {latest_file}")
    df = pd.read_csv(latest_file)
    
    print(f"\nData shape: {df.shape}")
    print(f"Temperature: {df['temperature'].mean():.2f} K")
    print(f"Bias: {df['bias'].mean():.3f} V")
    print(f"Frequency range: {df['frequency'].min():.0f} - {df['frequency'].max():.0f} Hz")
    print(f"\nFirst few rows:")
    print(df.head())
else:
    print("No data files found yet")

In [ ]:
# Plot impedance magnitude and phase vs frequency
if 'df' in locals():
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Impedance magnitude
    ax1.loglog(df['frequency'], df['Z'], 'b-', linewidth=2)
    ax1.set_xlabel('Frequency (Hz)', fontsize=12)
    ax1.set_ylabel('|Z| (Ω)', fontsize=12)
    ax1.set_title(f"Impedance at T={df['temperature'].mean():.1f}K, Bias={df['bias'].mean():.2f}V", fontsize=13)
    ax1.grid(True, alpha=0.3, which='both')
    
    # Phase
    ax2.semilogx(df['frequency'], df['theta'], 'r-', linewidth=2)
    ax2.set_xlabel('Frequency (Hz)', fontsize=12)
    ax2.set_ylabel('Phase θ (°)', fontsize=12)
    ax2.set_title(f"Phase at T={df['temperature'].mean():.1f}K, Bias={df['bias'].mean():.2f}V", fontsize=13)
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    print("Load data first (run cell above)")

In [ ]:
# Temperature + Bias sweep
temp_points = [300, 250, 200]  # K
bias_points = np.linspace(0, 1, 6)  # 0 to 1V in 0.2V steps

for target_temp in temp_points:
    print(f"\n{'='*50}")
    print(f"Temperature: {target_temp} K")
    print('='*50)
    
    janis.temperature_setpoint = target_temp
    print("Waiting for stability...")
    
    while not janis.temperature_stable:
        print(f"  Current: {janis.temperature:.1f} K")
        time.sleep(10)
    
    actual_temp = janis.temperature
    print(f"✓ Stable at {actual_temp:.1f} K")
    
    for bias in bias_points:
        print(f"\n  Bias: {bias:.2f} V")
        lcr.bias = bias
        time.sleep(0.5)
        
        # Filename format: run###_temp_###_bias_#.##.csv (matches your import regex)
        filename = f"C:/Users/F110216/Documents/DATA/Horatio/run{run_count}_temp_{actual_temp:.0f}_bias_{bias:.2f}.csv"
        
        with open(filename, "w") as f:
            f.write("# time,bias,frequency,NA,Z,theta\n")
            sweep_frequency_lcr(janis, lcr, frequency_points, f)
        
        print(f"  ✓ Saved: run{run_count}")
        run_count += 1

print(f"\n{'='*50}")
print(f"ALL DONE! Next run: {run_count}")
print('='*50)

In [ ]:
# Calculate capacitance and other parameters (optional)
if 'df' in locals():
    # For a parallel plate capacitor model
    # C = -1 / (2*pi*f*Z*sin(theta))
    # Or simpler: C = 1/(2*pi*f*Z) when theta ~ -90°
    
    df['C_calc'] = 1 / (2 * np.pi * df['frequency'] * df['Z'] * np.abs(np.sin(np.radians(df['theta']))))
    
    plt.figure(figsize=(10, 5))
    plt.loglog(df['frequency'], df['C_calc']*1e12, 'g-', linewidth=2)  # Convert to pF
    plt.xlabel('Frequency (Hz)', fontsize=12)
    plt.ylabel('Capacitance (pF)', fontsize=12)
    plt.title(f"Calculated Capacitance at T={df['temperature'].mean():.1f}K", fontsize=13)
    plt.grid(True, alpha=0.3, which='both')
    plt.show()
    
    print(f"Capacitance range: {df['C_calc'].min()*1e12:.2f} - {df['C_calc'].max()*1e12:.2f} pF")
else:
    print("Load data first")

---
## Quick Reference

**Enum Options:**
- `E4890A.MeasurementTime`: `SHORT`, `MEDIUM`, `LONG`
- `E4890A.MeasurementType`: `ZTD`, `ZTR`, `CPD`, `CPQ`, etc.
- `E4890A.SignalType`: `VOLTAGE`, `CURRENT`

**Current Settings Check:**

In [ ]:
# Check current instrument status
print("=== Janis Status ===")
print(f"Current temperature: {janis.temperature:.2f} K")
print(f"Setpoint: {janis.temperature_setpoint:.2f} K")
print(f"Stable: {janis.temperature_stable}")

print("\n=== LCR Meter Status ===")
print(f"Measurement type: {lcr.measurement_type.name}")
print(f"Signal: {lcr.signal_amplitude} V")
print(f"Averages: {lcr.averages}")
print(f"Bias: {lcr.bias} V")
print(f"Current frequency: {lcr.frequency} Hz")

print(f"\n=== Run Counter ===")
print(f"Next run number: {run_count}")

---
## Data Analysis & Plotting

Load and visualize your measurement data